# Week 62, Day 2: Identity and mutability

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
a = [1, 2]
b = [1, 2]
print(a == b, a is b)

t = (1, [2, 3])
t[1].append(4)
print(t)

try:
    {t: "value"}
except TypeError as e:
    print("TypeError:", e)

print(None is None, type(a) is list)

In [ ]:
a = [1, 2]
b = [1, 2]
c = a
print(a == b, a is b, a is c)
print(type(a) is type(b), id(a) == id(c))

nan = float("nan")
print(nan == nan, nan is nan, nan in [nan])

class Shouty:
    def __eq__(self, other):
        return True                        # claims to equal everything

s = Shouty()
print(s == 42, s == None, s is None)

In [ ]:
MISSING = object()

def lookup(table, key, default=MISSING):
    if key in table:
        return table[key]
    if default is MISSING:
        raise KeyError(key)
    return default

table = {"a": 1}
print(lookup(table, "a"), lookup(table, "b", None), lookup(table, "b", 0))
try:
    lookup(table, "b")
except KeyError as e:
    print("KeyError", e)

x = int("1000")
y = int("1000")
print(x == y)                              # the right test for numbers

In [ ]:
s = "tea"
t = s.upper()                             # a new string
print(s, t)

frozen = (1, 2, [3])
before = str(frozen)
frozen[2].append(4)                       # the tuple didn't change; the list did
print(before, frozen)

fs = frozenset({1, 2})
try:
    fs.add(3)
except AttributeError as e:
    print("AttributeError:", e)

In [ ]:
print(hash((1, "a")) == hash((1, "a")))
for value in ([1], {1: 2}, (1, [2])):
    try:
        hash(value)
    except TypeError as e:
        print("TypeError:", e)

class Tag:
    def __init__(self, name):
        self.name = name
    def __eq__(self, other):
        return isinstance(other, Tag) and self.name == other.name

print(Tag.__hash__ is None)              # defining __eq__ removed hashing

class BadKey:
    def __init__(self, n):
        self.n = n
    def __eq__(self, other):
        return isinstance(other, BadKey) and self.n == other.n
    def __hash__(self):
        return hash(self.n)               # hashed on a mutable field

k = BadKey(1)
d = {k: "found"}
k.n = 2                                   # the key's hash changed while in the dict
print(k in d, BadKey(1) in d, len(d))

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Money:
    pence: int
    currency: str = "GBP"

m = Money(250)
try:
    m.pence = 300
except AttributeError as e:
    print(type(e).__name__)
print(replace(m, pence=300), hash(m) == hash(Money(250)))

class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        object.__setattr__(self, "x", x)
        object.__setattr__(self, "y", y)

    def __setattr__(self, name, value):
        raise AttributeError("Point is immutable")

    __delattr__ = __setattr__

    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

    def with_x(self, x):
        return Point(x, self.y)

p = Point(1, 2)
try:
    p.x = 5
except AttributeError as e:
    print(e)
print({p: "home"}[Point(1, 2)], p.with_x(5).x, p.x)

> **Key idea.** Every object has a fixed identity (id, compared with is), a fixed type, and a value (compared with ==, which a class can define). Use is for None, sentinels and other singletons, and to check for aliasing; never for numbers or strings, whose caching is an implementation detail. Immutable types (int, str, tuple, frozenset, ...) can't change value; immutability is shallow, so a tuple holding a list can change. Hashing needs a hash that never changes and agrees with ==: mutable containers are unhashable, defining __eq__ removes the default hash, and __hash__ should use only immutable fields. Make value classes immutable with frozen dataclasses or a blocking __setattr__, and "change" them by returning new objects.

### Exercise: Hashable and deeply immutable

Write `is_hashable(obj)`: True if `hash(obj)` works. Then `deeply_immutable(obj)`: True if `obj` is one of `int`, `float`, `complex`, `bool`, `str`, `bytes`, `NoneType`, or a `tuple` or `frozenset` whose items are all deeply immutable (check the types; anything else is False, even if it's hashable).

In [ ]:
def is_hashable(obj):
    ___

def deeply_immutable(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_hashable():\n    \"hashability and deep immutability\"\n    assert is_hashable((1, \"a\")) and is_hashable(None) and is_hashable(frozenset({1}))\n    assert not is_hashable([1]) and not is_hashable((1, [2])) and not is_hashable({})\n    assert deeply_immutable((1, (\"a\", b\"x\", None), frozenset({2.5})))\n    assert not deeply_immutable((1, [2])) and not deeply_immutable([1])\n    class Plain:\n        pass\n    assert is_hashable(Plain()) and not deeply_immutable(Plain())\n    assert deeply_immutable(()) and deeply_immutable(True)")

### Exercise: A sentinel for no value

Write `pop_setting(settings, key, default=MISSING)` that removes `key` from the dict and returns its value; if the key is missing, it returns the default, or raises `KeyError(key)` when no default was given. `None` must work as a real default. Define `MISSING` as a unique sentinel object. Then `first_set(*values)`: the first value that isn't the sentinel `MISSING` (even if it's `None` or `0`), or `MISSING` if there are none.

In [ ]:
MISSING = ___

def pop_setting(settings, key, default=MISSING):
    ___

def first_set(*values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sentinel():\n    \"sentinels\"\n    s = {\"a\": 1, \"b\": None}\n    assert pop_setting(s, \"b\") is None and s == {\"a\": 1}\n    assert pop_setting(s, \"x\", None) is None and pop_setting(s, \"x\", 0) == 0\n    try:\n        pop_setting(s, \"x\")\n        assert False\n    except KeyError:\n        pass\n    assert first_set(MISSING, None, 3) is None and first_set(MISSING, 0) == 0\n    assert first_set() is MISSING and first_set(MISSING) is MISSING\n    assert MISSING is not None and MISSING != 0")

### Exercise: An immutable value class

Write a class `Colour(r, g, b)` (each 0 to 255, otherwise `ValueError`) that is immutable: setting or deleting any attribute raises `AttributeError`. Equal colours compare equal and hash equally, so they work as dict keys. `with_red(r)` (and the same for green and blue) returns a **new** colour, `hex()` gives `"#rrggbb"` in lower case, and `repr` is `Colour(255, 0, 0)`. Don't use a dataclass.

In [ ]:
class Colour:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_colour():\n    \"immutable values\"\n    red = Colour(255, 0, 0)\n    assert red == Colour(255, 0, 0) and red is not Colour(255, 0, 0)\n    assert {red: \"red\"}[Colour(255, 0, 0)] == \"red\" and len({red, Colour(255, 0, 0)}) == 1\n    for action in (lambda: setattr(red, \"r\", 1), lambda: delattr(red, \"r\"), lambda: setattr(red, \"alpha\", 1)):\n        try:\n            action()\n            assert False\n        except AttributeError:\n            pass\n    pink = red.with_blue(128)\n    assert pink.hex() == \"#ff0080\" and red.hex() == \"#ff0000\" and repr(pink) == \"Colour(255, 0, 128)\"\n    assert red.with_green(10).g == 10 and red.with_red(1).r == 1\n    assert red != (255, 0, 0)\n    try:\n        Colour(256, 0, 0)\n        assert False\n    except ValueError:\n        pass")

### Exercise: Same object or equal value

Write `unique_objects(items)`: the items with repeats of the **same object** removed (by identity, keeping the first), so two separate equal lists both stay. Then `unique_values(items)`: repeats of **equal** values removed (keeping the first), working even for unhashable items like lists (you may compare with `==`; don't use `set` on the items).

In [ ]:
def unique_objects(items):
    ___

def unique_values(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_identity():\n    \"identity versus equality\"\n    a, b = [1], [1]\n    out = unique_objects([a, b, a, b, a])\n    assert len(out) == 2 and out[0] is a and out[1] is b\n    assert unique_values([a, b, [2], a]) == [[1], [2]] and unique_values([a, b])[0] is a\n    assert unique_values([1, 1.0, True, \"1\"]) == [1, \"1\"]\n    nan = float(\"nan\")\n    assert len(unique_objects([nan, nan])) == 1\n    assert unique_values([]) == [] and unique_objects([]) == []")

### Exercise: Mutable parts inside

Write `mutable_parts(obj, path="obj")`: a list of path strings to every **mutable** container (list, dict, set) inside a nested structure, in depth-first order, including `obj` itself if it's mutable. Paths use indexing: `obj[1]`, `obj[2]['a']`. Look inside tuples, lists and dict values (sets can only hold hashable items, so don't look inside them). Then `freeze(obj)`: a deeply immutable copy, turning lists into tuples, sets into frozensets and dicts into tuples of `(key, value)` pairs sorted by key.

In [ ]:
def mutable_parts(obj, path="obj"):
    ___

def freeze(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mutable_parts():\n    \"shallow immutability\"\n    data = (1, [2], {\"a\": (3, [4]), \"b\": {5}})\n    assert mutable_parts(data) == [\"obj[1]\", \"obj[2]\", \"obj[2]['a'][1]\", \"obj[2]['b']\"]\n    assert mutable_parts((1, (\"x\",))) == [] and mutable_parts([], \"x\") == [\"x\"]\n    frozen = freeze(data)\n    assert frozen == (1, (2,), ((\"a\", (3, (4,))), (\"b\", frozenset({5}))))\n    assert mutable_parts(frozen) == [] and hash(frozen) == hash(freeze(data))\n    assert freeze({\"b\": 1, \"a\": [2]}) == ((\"a\", (2,)), (\"b\", 1))")

### Bug hunt: Equal to None

The `Wildcard` class claims to equal everything, so `== None` counts it as missing and this prints `2`. Count the real `None` values with an identity test so it prints `1`.

In [ ]:
class Wildcard:
    def __eq__(self, other):
        return True

values = [Wildcard(), None, 3]
missing = sum(1 for v in values if v == None)
print(missing)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"is None\"\n    expect_output(\"1\")")

### Bug hunt: Is it empty?

`is_empty` compares with `is`, asking whether `items` is the **same object** as a brand-new empty list, which is never true, so this prints `False False`. Make it test emptiness so it prints `True False`.

In [ ]:
def is_empty(items):
    return items is []

print(is_empty([]), is_empty([1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"values, not identity\"\n    expect_output(\"True False\")")

### Bug hunt: A key that moved

`Seat` hashes on its mutable `row`, so after the seat is moved its hash changes and the booking can't be found: this prints `False`. Make seats hash and compare by their `code` (which never changes) so it prints `True`.

In [ ]:
class Seat:
    def __init__(self, code, row):
        self.code, self.row = code, row
    def __eq__(self, other):
        return isinstance(other, Seat) and self.code == other.code
    def __hash__(self):
        return hash(self.row)

seat = Seat("A1", 1)
bookings = {seat: "Ada"}
seat.row = 7
print(seat in bookings)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stable hashes\"\n    expect_output(\"True\")")

### Puzzle: Equality and hashing together

Arrange the methods of a value class that compares and hashes on the same fields. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # [1] == [1], [1] is [1] as a tuple
q = ___   # True or False: can (1, [2]) be a dict key?
r = ___   # True or False: is a class with __eq__ but no __hash__ hashable?
s = ___   # float("nan") == float("nan")
t = ___   # t = (1, [2]); t[1].append(3); len(t)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (True, False) and q is False and r is False, \"p, q and r\"\n    assert s is False and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a frozen dictionary

Write `FrozenDict`, an immutable, hashable mapping. Make it a subclass of `collections.abc.Mapping` (Week 33) so it gets `keys`, `items`, `values`, `get`, `in` and `==` for free. It takes the same arguments as `dict(...)`; item assignment and deletion fail with `TypeError` (they're not defined); setting attributes from outside fails. `hash` works when every value is hashable (compute it once, from a `frozenset` of the items, and cache it) and is the same for equal FrozenDicts. `set(key, value)` and `remove(key)` return **new** FrozenDicts. `repr` is `FrozenDict({'a': 1})`. Then `deep_freeze(obj)`, which turns dicts into FrozenDicts, lists into tuples and sets into frozensets, all the way down, and `thaw(obj)`, which turns them back.

In [ ]:
from collections.abc import Mapping

class FrozenDict(Mapping):
    ___

def deep_freeze(obj):
    ___

def thaw(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frozendict():\n    \"an immutable mapping\"\n    f = FrozenDict({\"a\": 1}, b=2)\n    assert f[\"a\"] == 1 and len(f) == 2 and \"b\" in f and f.get(\"z\") is None\n    assert f == {\"a\": 1, \"b\": 2} and f == FrozenDict(b=2, a=1)\n    assert hash(f) == hash(FrozenDict(b=2, a=1)) and {f: \"yes\"}[FrozenDict(a=1, b=2)] == \"yes\"\n    g = f.set(\"c\", 3)\n    assert g[\"c\"] == 3 and \"c\" not in f and f.remove(\"a\") == {\"b\": 2} and \"a\" in f\n    for action in (lambda: f.__setitem__(\"a\", 5), lambda: f.__delitem__(\"a\")):\n        try:\n            action()\n            assert False\n        except (TypeError, AttributeError):\n            pass\n    try:\n        f[\"a\"] = 5\n        assert False\n    except TypeError:\n        pass\n    try:\n        f.extra = 1\n        assert False\n    except AttributeError:\n        pass\n    assert repr(FrozenDict(a=1)) == \"FrozenDict({'a': 1})\"\n    source = {\"a\": 1}\n    h = FrozenDict(source)\n    source[\"a\"] = 99\n    assert h[\"a\"] == 1\n\ndef test_deep():\n    \"deep freeze and thaw\"\n    data = {\"name\": \"atlas\", \"tags\": [\"py\", \"free\"], \"meta\": {\"weeks\": {104}}}\n    frozen = deep_freeze(data)\n    assert isinstance(frozen, FrozenDict) and frozen[\"tags\"] == (\"py\", \"free\")\n    assert isinstance(frozen[\"meta\"], FrozenDict) and frozen[\"meta\"][\"weeks\"] == frozenset({104})\n    assert hash(frozen) == hash(deep_freeze(data))\n    assert thaw(frozen) == data and type(thaw(frozen)[\"meta\"]) is dict and thaw(frozen)[\"tags\"] == [\"py\", \"free\"]")

## Warm-up

Equal lists are still two objects; a tuple is immutable, but the list inside it isn't; and a tuple holding a list can't be a dict key, because its value could change.

## Identity, type and value

Every object has three things:

- an **identity**, fixed for its whole life: `id(obj)`, compared with `is`;
- a **type**, also fixed: `type(obj)`;
- a **value**, compared with `==` (which calls `__eq__`).

`a is b` asks "are these the **same object**?" and can't be overridden. `a == b` asks "do these have **equal values**?" and is whatever the class's `__eq__` says. Equal objects needn't be identical (two lists `[1, 2]`), and, oddly, an object needn't even equal itself: `float("nan") == float("nan")` is `False`. (Containers check identity first when testing membership, which is why `nan in [nan]` is `True` for the very same `nan` object.)

## When to use is

Use `is` only when identity is genuinely the question:

- **`None`**: always `x is None` / `x is not None`. There is exactly one `None`, and `is` can't be fooled by a strange `__eq__` (like `Shouty` above).
- **Sentinels**: a unique marker object, `MISSING = object()`, for "no value given" when `None` is itself a valid value. Nothing else can ever be that object.
- **`True`, `False`, `NotImplemented`, `Ellipsis`**: also singletons, though for booleans plain truth tests are usually better.
- **Checking for aliasing**: "is this the same list I already have?"

Never use `is` to compare numbers or strings. CPython **caches** some small objects (integers from -5 to 256, many short strings), so `x is 5` may happen to work and then fail for 1000 or for a string built at runtime. That caching is an implementation detail that varies between versions; Python even warns when it sees `is` with a literal.

## Mutable and immutable types

**Immutable** objects can't change their value after creation: `int`, `float`, `complex`, `bool`, `str`, `bytes`, `tuple`, `frozenset`, `range`, `None`. Operations that seem to change them build new objects. **Mutable** objects can: `list`, `dict`, `set`, `bytearray`, and instances of ordinary classes.

Immutability is **shallow**: a tuple can't change which objects it holds, but if one of those objects is a list, the list can change, and so the tuple's *value* (what it prints, what it equals) changes too. A tuple is "deeply immutable" only if everything inside it is.

Why immutability is valuable: an immutable object can be **shared freely** (no one can change it under you), used as a **dict key or set member**, and reasoned about locally. That's why Python makes strings immutable, and why it's worth making your own value types (money, coordinates, dates) immutable.

## Hashing needs immutability

A dict finds a key by its **hash**, then confirms with `==` (Week 44). That only works if a key's hash **never changes** while it's in the dict. So the rule for any class is:

- objects that are **equal must have equal hashes** (`a == b` implies `hash(a) == hash(b)`);
- the hash must stay the same for the object's life, which in practice means it must depend only on **immutable** parts.

Python's built-ins follow this: mutable containers are **unhashable** (`hash([])` raises `TypeError`), and a tuple is hashable only if everything in it is. For your own classes: by default instances hash by identity and equal only themselves. If you define `__eq__`, Python sets `__hash__` to `None` (unhashable), because the default identity hash would break the rule; define `__hash__` too, based on the same fields as `__eq__`, **only** if those fields never change.

## Writing immutable classes

Python has no true lock on an object, but you can make accidental changes fail:

- **Frozen dataclasses** (Week 33): `@dataclass(frozen=True)` writes `__init__`, `__eq__` and `__hash__`, and makes assignment raise `FrozenInstanceError` (a kind of `AttributeError`).
- **By hand**: set fields once in `__init__` with `object.__setattr__`, and override `__setattr__` and `__delattr__` to raise `AttributeError`. `__slots__` (Week 31) stops new attributes appearing.
- **"Changing" means making a new one**: methods like `with_x(5)` return a fresh object, like `str.replace` or `dataclasses.replace`.
- Keep the fields themselves immutable: store a `tuple(items)`, not the caller's list.

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at a class you wrote in Phase 3 that defines `__eq__`. Is it hashable? Should it be? Tomorrow: what happens to objects nobody refers to any more, with reference counting and the garbage collector.